# Reproduce the FireSight FLEX model
P(droplet flame keeps burning) from O₂, ln(pressure), CO₂, He and fuel, with 5-fold cross-validation.
Requires `pandas`, `numpy`, `scikit-learn`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss

df = pd.read_csv('firesight_tests.csv')
flex = df[(df.investigation == 'PSI-69') & df.o2_pct.notna() & df.pressure_kpa.notna()].copy()
len(flex)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X = np.column_stack([flex.o2_pct, np.log(flex.pressure_kpa), flex.co2_pct.fillna(0), flex.he_pct.fillna(0), (flex.fuel == 'Heptane').astype(int)])
y = (flex.outcome != 'flame extinction').astype(int).values
cv = StratifiedKFold(5, shuffle=True, random_state=7)  # same folds as the app
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000))  # scaler fitted inside each fold
p = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
print('AUC', round(roc_auc_score(y, p), 3), 'Brier', round(brier_score_loss(y, p), 3), 'no-skill', round(brier_score_loss(y, np.full_like(p, y.mean())), 3))

In [ ]:
m = pipe.fit(X, y)[-1]
dict(zip(['o2', 'ln_p', 'co2', 'he', 'heptane'], m.coef_[0].round(3)))

Expected output: AUC 0.859, Brier 0.138, no-skill 0.219, identical to the app. The app adds 300 bootstrap refits for its 90% intervals.